### Моделирование v2 (КОМБИНИРОВАННЫЙ ТАРГЕТ)

**Исправления:**
1. Таргет: активность = (train >= 3) OR (val >= 3) OR (commits > 0)
2. Исключены признаки с утечкой (jira_issues_count и производные)
3. SMOTE для балансировки классов
4. Оптимальный порог классификации по F1

## 0. Настройка окружения

In [ ]:
import sys
sys.path.append('..')

import warnings
warnings.filterwarnings('ignore')

import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.preprocessing import StandardScaler, LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LogisticRegression
from sklearn.ensemble import RandomForestClassifier, GradientBoostingClassifier, VotingClassifier
from sklearn.neighbors import KNeighborsClassifier
from sklearn.metrics import (
    roc_auc_score, precision_recall_curve, auc,
    precision_score, recall_score, f1_score, average_precision_score
)
from imblearn.over_sampling import SMOTE

from src.config import RANDOM_SEED

print('Импорты готовы')

## 1. Загрузка данных

In [ ]:
from src.features.profiles import build_developer_profiles, add_advanced_features
from src.data.loader import load_data
from src.data.cleaner import clean_data

employees = pd.read_csv('../data/processed/employees_clean.csv')
jira_train = pd.read_csv('../data/processed/jira_train.csv')
jira_val = pd.read_csv('../data/processed/jira_val.csv')

_, _, gitlab = load_data('../data/raw')
_, _, gitlab = clean_data(employees.copy(), jira_train.copy(), gitlab.copy())

print(f'Employees: {employees.shape}')
print(f'Jira train: {jira_train.shape}')
print(f'Jira val: {jira_val.shape}')
print(f'Gitlab: {gitlab.shape}')

## 2. Комбинированный таргет

In [ ]:
# Активность на train
train_activity = jira_train.groupby('assignee_login_nm').size().reset_index(name='train_tasks')
train_activity.columns = ['login', 'train_tasks']

# Активность на validation
val_activity = jira_val.groupby('assignee_login_nm').size().reset_index(name='val_tasks')
val_activity.columns = ['login', 'val_tasks']

# Активность в gitlab (коммиты)
git_activity = gitlab.groupby('author_login').size().reset_index(name='git_commits')
git_activity.columns = ['login', 'git_commits']

# Объединение
profiles_base = employees[['login']].copy()
profiles_base = profiles_base.merge(train_activity, on='login', how='left')
profiles_base = profiles_base.merge(val_activity, on='login', how='left')
profiles_base = profiles_base.merge(git_activity, on='login', how='left')

profiles_base['train_tasks'] = profiles_base['train_tasks'].fillna(0).astype(int)
profiles_base['val_tasks'] = profiles_base['val_tasks'].fillna(0).astype(int)
profiles_base['git_commits'] = profiles_base['git_commits'].fillna(0).astype(int)

# КОМБИНИРОВАННЫЙ ТАРГЕТ
profiles_base['is_active'] = (
    (profiles_base['train_tasks'] >= 3) |
    (profiles_base['val_tasks'] >= 3) |
    (profiles_base['git_commits'] > 0)
).astype(int)

print('=== Распределение таргета ===')
print(profiles_base['is_active'].value_counts())
print(f'\nДисбаланс: {profiles_base["is_active"].mean():.2%} активных')

print('\n=== Компоненты таргета ===')
print(f'train_tasks >= 3: {(profiles_base["train_tasks"] >= 3).sum()}')
print(f'val_tasks >= 3: {(profiles_base["val_tasks"] >= 3).sum()}')
print(f'git_commits > 0: {(profiles_base["git_commits"] > 0).sum()}')
print(f'Все нули: {((profiles_base["train_tasks"] == 0) & (profiles_base["val_tasks"] == 0) & (profiles_base["git_commits"] == 0)).sum()}')

## 3. Построение профилей и признаков

In [ ]:
# Профили (без утечек - только train данные для признаков)
profiles = build_developer_profiles(employees, jira_train, gitlab, jira_train=jira_train)
profiles = add_advanced_features(profiles)

# Добавляем таргет
profiles = profiles.merge(
    profiles_base[['login', 'train_tasks', 'val_tasks', 'git_commits', 'is_active']],
    on='login', how='left'
)

print(f'Профили: {profiles.shape}')
print(f'Таргет: {profiles["is_active"].mean():.2%} активных')

In [ ]:
# Признаки
safe_features = [
    'work_experience_day_cnt',
    'commits_count',
    'projects_count', 
    'repos_count',
    'profile_words_count',
    'commits_per_experience',
    'versatility_score',
    'text_density',
    'is_senior',
    'is_junior',
    'log_experience',
    'log_commits_count'
]

# Кодирование категориальных
le_spec = LabelEncoder()
profiles['specialization_encoded'] = le_spec.fit_transform(profiles['specialization_nm'].fillna('Unknown'))
safe_features.append('specialization_encoded')

le_pos = LabelEncoder()
profiles['position_encoded'] = le_pos.fit_transform(profiles['position_nm'].fillna('Unknown'))
safe_features.append('position_encoded')

X = profiles[safe_features].fillna(0).values
y = profiles['is_active'].values

# Масштабирование
scaler = StandardScaler()
X_scaled = scaler.fit_transform(X)

# Разделение на train/test
X_train, X_test, y_train, y_test = train_test_split(
    X_scaled, y, test_size=0.2, random_state=RANDOM_SEED, stratify=y
)

print(f'Признаки: {len(safe_features)}')
print(f'Train: {len(X_train)}, Test: {len(X_test)}')
print(f'Дисбаланс train: {y_train.mean():.2%}')

## 4. Моделирование с SMOTE

In [ ]:
def find_optimal_threshold(y_true, y_proba):
    precision, recall, thresholds = precision_recall_curve(y_true, y_proba)
    f1_scores = 2 * (precision * recall) / (precision + recall + 1e-10)
    best_idx = np.argmax(f1_scores)
    return thresholds[best_idx] if best_idx < len(thresholds) else 0.5

def ranking_metrics(y_true, y_score, k=5):
    sorted_indices = np.argsort(y_score)[::-1]
    top_k = sorted_indices[:k]
    relevant_in_top_k = np.sum(y_true[top_k] == 1)
    total_relevant = np.sum(y_true == 1)
    return {
        'precision@5': relevant_in_top_k / k,
        'recall@5': relevant_in_top_k / total_relevant if total_relevant > 0 else 0.0
    }

# SMOTE
smote = SMOTE(random_state=RANDOM_SEED, k_neighbors=5)
X_train_resampled, y_train_resampled = smote.fit_resample(X_train, y_train)

print(f'После SMOTE: {len(X_train_resampled)} образцов (было {len(X_train)})')
print(f'Баланс: {np.mean(y_train_resampled):.2%}')

In [ ]:
models = {
    'Logistic Regression': LogisticRegression(random_state=RANDOM_SEED, max_iter=1000, class_weight='balanced'),
    'Random Forest': RandomForestClassifier(n_estimators=100, max_depth=10, random_state=RANDOM_SEED, n_jobs=-1, class_weight='balanced'),
    'Gradient Boosting': GradientBoostingClassifier(n_estimators=100, max_depth=5, random_state=RANDOM_SEED),
    'KNN': KNeighborsClassifier(n_neighbors=10, weights='distance'),
}

results = []
trained_models = {}

for name, model in models.items():
    print(f'\n=== {name} ===')
    
    model.fit(X_train_resampled, y_train_resampled)
    y_pred_proba = model.predict_proba(X_test)[:, 1]
    
    optimal_threshold = find_optimal_threshold(y_test, y_pred_proba)
    y_pred = (y_pred_proba >= optimal_threshold).astype(int)
    
    pr_auc = average_precision_score(y_test, y_pred_proba)
    roc_auc = roc_auc_score(y_test, y_pred_proba)
    precision = precision_score(y_test, y_pred, zero_division=0)
    recall = recall_score(y_test, y_pred, zero_division=0)
    f1 = f1_score(y_test, y_pred, zero_division=0)
    ranking = ranking_metrics(y_test, y_pred_proba, k=5)
    
    results.append({
        'model': name,
        'pr_auc': pr_auc,
        'roc_auc': roc_auc,
        'precision': precision,
        'recall': recall,
        'f1': f1,
        'optimal_threshold': optimal_threshold,
        'precision@5': ranking['precision@5'],
        'recall@5': ranking['recall@5']
    })
    
    trained_models[name] = model
    
    print(f'PR-AUC: {pr_auc:.4f}, ROC-AUC: {roc_auc:.4f}')
    print(f'Precision: {precision:.4f}, Recall: {recall:.4f}, F1: {f1:.4f}')
    print(f'Threshold: {optimal_threshold:.3f}, Recall@5: {ranking["recall@5"]:.4f}')

results_df = pd.DataFrame(results)
print('\n' + '='*60)
print('СВОДНАЯ ТАБЛИЦА (по PR-AUC)')
print('='*60)
print(results_df.sort_values('pr_auc', ascending=False).to_string(index=False))

## 5. Ансамбль

In [ ]:
voting_clf = VotingClassifier(
    estimators=[
        ('rf', trained_models['Random Forest']),
        ('gb', trained_models['Gradient Boosting']),
        ('lr', trained_models['Logistic Regression'])
    ],
    voting='soft'
)

voting_clf.fit(X_train_resampled, y_train_resampled)
y_pred_proba = voting_clf.predict_proba(X_test)[:, 1]

optimal_threshold = find_optimal_threshold(y_test, y_pred_proba)
y_pred = (y_pred_proba >= optimal_threshold).astype(int)

pr_auc = average_precision_score(y_test, y_pred_proba)
roc_auc = roc_auc_score(y_test, y_pred_proba)
f1 = f1_score(y_test, y_pred, zero_division=0)
ranking = ranking_metrics(y_test, y_pred_proba, k=5)

print('=== Voting Ensemble ===')
print(f'PR-AUC: {pr_auc:.4f}, ROC-AUC: {roc_auc:.4f}')
print(f'F1: {f1:.4f}, Recall@5: {ranking["recall@5"]:.4f}')

results.append({
    'model': 'Voting Ensemble',
    'pr_auc': pr_auc,
    'roc_auc': roc_auc,
    'precision': precision_score(y_test, y_pred, zero_division=0),
    'recall': recall_score(y_test, y_pred, zero_division=0),
    'f1': f1,
    'optimal_threshold': optimal_threshold,
    'precision@5': ranking['precision@5'],
    'recall@5': ranking['recall@5']
})

## 6. Важность признаков

In [ ]:
rf_model = trained_models.get('Random Forest')

importance_df = pd.DataFrame({
    'feature': safe_features,
    'importance': rf_model.feature_importances_
}).sort_values('importance', ascending=False)

plt.figure(figsize=(10, 8))
plt.barh(importance_df['feature'].head(10), importance_df['importance'].head(10), color='steelblue')
plt.xlabel('Важность признака')
plt.title('Топ-10 важных признаков (Random Forest)')
plt.gca().invert_yaxis()
plt.grid(axis='x', alpha=0.3)
plt.tight_layout()
plt.show()

print('=== Топ-10 важных признаков ===')
print(importance_df.head(10).to_string(index=False))

## 7. Сохранение модели

In [ ]:
import joblib

best_model_idx = results_df['pr_auc'].idxmax()
best_model_name = results_df.loc[best_model_idx, 'model']
best_model = trained_models[best_model_name]

joblib.dump(best_model, '../models/best_model_v2.joblib')
joblib.dump(scaler, '../models/scaler_v2.joblib')
joblib.dump(le_spec, '../models/label_encoder_spec_v2.joblib')
joblib.dump(le_pos, '../models/label_encoder_pos_v2.joblib')
results_df.to_csv('../data/processed/model_comparison_v2.csv', index=False)

print(f'Лучшая модель: {best_model_name}')
print(f'PR-AUC: {results_df.loc[best_model_idx, "pr_auc"]:.4f}')
print(f'F1: {results_df.loc[best_model_idx, "f1"]:.4f}')
print(f'Recall@5: {results_df.loc[best_model_idx, "recall@5"]:.4f}')
print('\nМодели сохранены в models/best_model_v2.joblib')